In [5]:
import os
import json
import pandas as pd

from io import BytesIO
from datetime import datetime
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

# 1. CARREGAR CREDENCIAL

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

if connection_string:
    print("Credencial carregada com sucesso!")
else:
    raise ValueError("Credencial não encontrada.")

# 2. CONECTAR AO AZURE

blob_service = BlobServiceClient.from_connection_string(
    connection_string
)

container_landing = blob_service.get_container_client("landing")
container_bronze = blob_service.get_container_client("bronze")

print("Conexão com Azure realizada com sucesso!")

# 3. LOCALIZAR ARQUIVO NA LANDING

arquivos = container_landing.list_blobs(
    name_starts_with="logistica/entrada/"
)

caminho_arquivo = None

for arquivo in arquivos:
    if arquivo.name.lower().endswith(".csv"):
        caminho_arquivo = arquivo.name
        break

# 4. PROCESSAR ARQUIVO

if caminho_arquivo is None:

    print("Nenhum arquivo CSV encontrado.")

else:

    nome_arquivo = os.path.basename(caminho_arquivo)

    blob_entrada = container_landing.get_blob_client(
        caminho_arquivo
    )

    dados = blob_entrada.download_blob().readall()

# 5. VALIDAÇÃO TÉCNICA

    arquivo_valido = True
    motivo = ""

    # Arquivo com zero bytes
    if len(dados) == 0:

        arquivo_valido = False
        motivo = "Arquivo vazio."

    else:

        try:

            df_teste = pd.read_csv(
                BytesIO(dados)
            )

            # CSV sem registros
            if df_teste.empty:

                arquivo_valido = False
                motivo = "Arquivo CSV sem registros."

        except Exception as erro:

            arquivo_valido = False
            motivo = f"Arquivo CSV inválido: {erro}"

# 6. ARQUIVO INVÁLIDO

    if not arquivo_valido:

        print("Arquivo rejeitado:", motivo)

        registro_erro = {
            "arquivo": nome_arquivo,
            "status": "erro",
            "motivo": motivo,
            "origem": caminho_arquivo,
            "data_processamento": datetime.now().isoformat()
        }

        nome_log_erro = nome_arquivo.replace(
            ".csv",
            ".json"
        )

        blob_erro = container_landing.get_blob_client(
            f"logistica/erros/{nome_log_erro}"
        )

        blob_erro.upload_blob(
            json.dumps(
                registro_erro,
                indent=4
            ),
            overwrite=True
        )

        print("Registro de erro criado.")
        print("Arquivo mantido na Landing/entrada para análise.")

# 7. ARQUIVO VÁLIDO

    else:

        print("Arquivo validado:", nome_arquivo)

        # Identificar ano
        ano = nome_arquivo.split("_")[1]

        destino = f"logistica/{ano}/{nome_arquivo}"

# 8. ENVIAR PARA BRONZE

        blob_bronze = container_bronze.get_blob_client(
            destino
        )

        blob_bronze.upload_blob(
            dados,
            overwrite=False
        )

        print("Arquivo enviado para Bronze.")

# 9. REGISTRAR PROCESSAMENTO

        registro = {
            "arquivo": nome_arquivo,
            "status": "sucesso",
            "origem": caminho_arquivo,
            "destino": destino,
            "data_processamento": datetime.now().isoformat()
        }

        nome_log = nome_arquivo.replace(
            ".csv",
            ".json"
        )

        blob_log = container_landing.get_blob_client(
            f"logistica/processados/{nome_log}"
        )

        blob_log.upload_blob(
            json.dumps(
                registro,
                indent=4
            ),
            overwrite=False
        )

        print("Registro de processamento criado.")

# 10. REMOVER DA ENTRADA

        blob_entrada.delete_blob()

        print("Arquivo removido da Landing/entrada.")


# 11. RESULTADO

        print("\nIngestão concluída com sucesso!")
        print("Arquivo:", nome_arquivo)
        print("Destino:", destino)

Credencial carregada com sucesso!
Conexão com Azure realizada com sucesso!
Arquivo validado: entregas_2025_T3.csv
Arquivo enviado para Bronze.
Registro de processamento criado.
Arquivo removido da Landing/entrada.

Ingestão concluída com sucesso!
Arquivo: entregas_2025_T3.csv
Destino: logistica/2025/entregas_2025_T3.csv
